# Sanity 02 — the field-split tokenizer

Mirrors `scripts/02_tokenize.py`. The core idea of the whole template: **one position per
transaction**. Static card fields are stored once per window (`s_*`); each dynamic field is its
own token column (`d_*`) of length `seq_len`; windows are **right-aligned** (left-padded), so
the LAST position is the target transaction. Continuous channels (`d_amount_log`,
`d_delta_log`) ride alongside the bucketed tokens for the model's periodic (Fourier/Time2Vec)
inputs.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))   # template root (src/)
BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
SCALE = "full"                               # flip to "xl" / "xxl" to inspect those runs

In [ ]:
import pyarrow.dataset as pads

ev = pads.dataset(f"{BASE}/tokenized/{SCALE}/eval", format="parquet")
print(f"{ev.count_rows():,} eval windows")
print([n for n in ev.schema.names])

In [ ]:
row = ev.head(1).to_pandas().iloc[0]
seq_len = len(np.asarray(row["attention_mask"]))
n_real = int(np.asarray(row["attention_mask"]).sum())
print(f"seq_len={seq_len}, real txns in this window={n_real} (rest is left-padding)")
print("statics:", {c: int(row[c]) for c in row.index if c.startswith("s_")})
print("label:", row["label"], "| raw_amount:", row["raw_amount"], "| raw_ts:", row["raw_ts"])

In [ ]:
# The last 8 positions of each dynamic field — position -1 IS the target transaction.
for c in row.index:
    if c.startswith("d_") and not c.endswith(("_log",)):
        print(f"{c:>22}: {np.asarray(row[c])[-8:].tolist()}")
print(f"{'d_amount_log (cont.)':>22}: {np.round(np.asarray(row['d_amount_log'])[-8:], 3).tolist()}")
print(f"{'time_bucket (gap)':>22}: {np.asarray(row['time_bucket'])[-8:].tolist()}")

In [ ]:
# Cross-check: the window's target matches a benchmark row on (card_id, ts, amount-cents).
bench = pd.read_parquet(f"{BASE}/raw/{SCALE}/benchmark.parquet")
bench["_amt_cents"] = np.round(bench["Amount"].to_numpy() * 100).astype(np.int64)
hit = bench[(bench.card_id == row["card_id"]) & (bench._ts == row["raw_ts"]) &
            (bench._amt_cents == round(float(row["raw_amount"]) * 100))]
print("matching benchmark rows:", len(hit)); hit[["split", "_target", "Amount", "Merchant City"]]

**What to check:** token ids 0/1/2 are PAD/MASK special ids; padding sits on the LEFT;
`n_real` ≤ seq_len (cards with short history get partial windows — the model handles that via
the attention mask); the benchmark cross-check should return exactly one row.